# Philadelphia Transit-Oriented Communities Political Analysis

Analysis of transit stations by City Council district for TOC legislation advocacy.

## Objectives
1. Count transit stations per City Council district
2. Identify high-ridership stations
3. Assess market conditions for development likelihood
4. Generate political briefing materials

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from pathlib import Path
import numpy as np

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

# Define paths
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
OUTPUTS = Path('../outputs')

print('✓ Libraries loaded successfully')

## 1. Load Data

In [ ]:
# Load City Council districts
council_districts = gpd.read_file(DATA_RAW / 'city_council_districts.geojson')
print(f"Loaded {len(council_districts)} City Council districts")
print(f"CRS: {council_districts.crs}")
print(f"Columns: {list(council_districts.columns)}")
council_districts.head()

In [ ]:
# Load all transit stops
all_stops = gpd.read_file(DATA_RAW / 'septa_all_transit_stops.geojson')
print(f"Loaded {len(all_stops)} total transit stops")
print(f"CRS: {all_stops.crs}")
print(f"Columns: {list(all_stops.columns)}")

# Filter to rail stations only (exclude bus stops)
if 'type' in all_stops.columns:
    rail_stops = all_stops[all_stops['type'].str.contains('Rail|rail|BSL|MFL|Broad|Market', na=False, case=False)]
elif 'route_id' in all_stops.columns:
    # Filter by route types - BSL (Broad Street Line), MFL (Market-Frankford), Regional Rail
    rail_stops = all_stops[all_stops['route_id'].str.contains('BSL|MFL|Regional|RR|TRN', na=False, case=False)]
else:
    # If no clear type field, we'll examine the data structure
    print("\nSample records:")
    print(all_stops.head())
    rail_stops = all_stops  # Use all for now, will refine

print(f"\nFiltered to {len(rail_stops)} rail stations")

In [ ]:
# Load Regional Rail stations separately (these are key stations)
regional_rail = gpd.read_file(DATA_RAW / 'septa_regional_rail_stations.geojson')
print(f"Loaded {len(regional_rail)} Regional Rail stations")
print(f"Columns: {list(regional_rail.columns)}")
regional_rail.head()

In [ ]:
# Load ridership data
ridership = pd.read_csv(DATA_RAW / 'septa_ridership_rail.csv')
print(f"Loaded ridership data: {ridership.shape}")
print(f"Columns: {list(ridership.columns)}")
print(f"\nYear range: {ridership.columns[ridership.columns.str.contains('20', na=False)].tolist() if any(ridership.columns.str.contains('20', na=False)) else 'Check column names'}")
ridership.head()

## 2. Data Preparation & Spatial Join

In [ ]:
# Ensure same CRS for spatial operations
if council_districts.crs != regional_rail.crs:
    regional_rail = regional_rail.to_crs(council_districts.crs)
    print(f"Reprojected Regional Rail to {council_districts.crs}")

# Perform spatial join: which district is each station in?
stations_with_districts = gpd.sjoin(
    regional_rail, 
    council_districts, 
    how='left', 
    predicate='within'
)

print(f"Spatial join complete: {len(stations_with_districts)} stations matched")
print(f"\nStations by district:")
print(stations_with_districts.groupby('DISTRICT').size().sort_index())

In [ ]:
# Create station name standardization for joining with ridership
# This will vary based on actual column names in the data
station_col = [col for col in regional_rail.columns if 'name' in col.lower() or 'station' in col.lower()]
print(f"Station name column(s): {station_col}")

ridership_station_col = [col for col in ridership.columns if 'station' in col.lower() or 'stop' in col.lower()]
print(f"Ridership station column(s): {ridership_station_col}")

## 3. Stations Per District Analysis

In [ ]:
# Count stations by district
district_counts = stations_with_districts.groupby('DISTRICT').agg({
    'OBJECTID': 'count'  # Adjust based on actual ID column
}).rename(columns={'OBJECTID': 'num_stations'}).reset_index()

# Get councilmember names if available
district_info = council_districts[['DISTRICT', 'COUNCIL_MEMBER']].drop_duplicates() \
    if 'COUNCIL_MEMBER' in council_districts.columns else \
    council_districts[['DISTRICT']].drop_duplicates()

# Merge
district_summary = district_info.merge(district_counts, on='DISTRICT', how='left')
district_summary['num_stations'] = district_summary['num_stations'].fillna(0).astype(int)
district_summary = district_summary.sort_values('DISTRICT')

print("\n=== TRANSIT STATIONS PER CITY COUNCIL DISTRICT ===")
print(district_summary.to_string(index=False))

# Save to CSV
district_summary.to_csv(OUTPUTS / 'stations_per_district.csv', index=False)
print("\n✓ Saved to outputs/stations_per_district.csv")

In [ ]:
# Visualization: Bar chart of stations per district
fig, ax = plt.subplots(figsize=(12, 6))
district_summary.plot.bar(x='DISTRICT', y='num_stations', ax=ax, color='#0066cc', legend=False)
ax.set_xlabel('City Council District', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Regional Rail Stations', fontsize=12, fontweight='bold')
ax.set_title('SEPTA Regional Rail Stations by Philadelphia City Council District', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticklabels(district_summary['DISTRICT'], rotation=0)
plt.tight_layout()
plt.savefig(OUTPUTS / 'stations_per_district_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved chart to outputs/stations_per_district_chart.png")

## 4. High Ridership Station Analysis

In [ ]:
# Identify most recent year column in ridership data
year_cols = [col for col in ridership.columns if col.startswith('20') or 'ridership' in col.lower()]
print(f"Available ridership columns: {year_cols}")

# Calculate average recent ridership (exclude COVID years 2020-2021)
recent_years = [col for col in year_cols if '2022' in col or '2023' in col or '2024' in col]
if recent_years:
    ridership['avg_ridership'] = ridership[recent_years].mean(axis=1)
    print(f"\nCalculated average ridership from: {recent_years}")
else:
    # Fallback to most recent single year
    recent_years = [year_cols[-1]] if year_cols else []
    if recent_years:
        ridership['avg_ridership'] = ridership[recent_years[0]]
        print(f"\nUsing ridership from: {recent_years[0]}")

# Show top ridership stations
if 'avg_ridership' in ridership.columns:
    top_ridership = ridership.nlargest(20, 'avg_ridership')
    print("\n=== TOP 20 HIGHEST RIDERSHIP STATIONS ===")
    print(top_ridership[ridership_station_col + ['avg_ridership']].to_string(index=False) 
          if ridership_station_col else top_ridership.head())

In [ ]:
# Merge ridership with spatial station data
# This requires matching station names - may need fuzzy matching
if ridership_station_col and station_col:
    # Create standardized names for matching
    ridership_key = ridership_station_col[0]
    station_key = station_col[0]
    
    ridership['station_std'] = ridership[ridership_key].str.upper().str.strip()
    stations_with_districts['station_std'] = stations_with_districts[station_key].str.upper().str.strip()
    
    # Merge
    stations_ridership = stations_with_districts.merge(
        ridership[['station_std', 'avg_ridership'] + recent_years],
        on='station_std',
        how='left'
    )
    
    # Identify high ridership stations (top quartile)
    ridership_threshold = stations_ridership['avg_ridership'].quantile(0.75)
    stations_ridership['high_ridership'] = stations_ridership['avg_ridership'] > ridership_threshold
    
    print(f"\nHigh ridership threshold: {ridership_threshold:,.0f} daily boardings")
    print(f"Number of high ridership stations: {stations_ridership['high_ridership'].sum()}")
    
    # High ridership stations by district
    high_ridership_by_district = stations_ridership[stations_ridership['high_ridership']].groupby('DISTRICT').size()
    print("\n=== HIGH RIDERSHIP STATIONS BY DISTRICT ===")
    print(high_ridership_by_district.to_string())

## 5. Market Conditions Analysis

For development likelihood, we'll create a simplified assessment based on:
- Station ridership (higher = more demand)
- District characteristics (if available)
- Distance from Center City (closer = higher land values)

In [ ]:
# Create Center City reference point (City Hall: ~39.9526° N, 75.1652° W)
from shapely.geometry import Point

city_hall = Point(-75.1652, 39.9526)

# Calculate distance from Center City for each station
if stations_ridership.crs.to_string() != 'EPSG:4326':
    stations_temp = stations_ridership.to_crs('EPSG:4326')
else:
    stations_temp = stations_ridership.copy()

# Distance in miles (approximate)
stations_ridership['dist_from_cc_miles'] = stations_temp.geometry.distance(city_hall) * 69  # degrees to miles approx

print("Distance from Center City calculated")
print(f"Range: {stations_ridership['dist_from_cc_miles'].min():.1f} - {stations_ridership['dist_from_cc_miles'].max():.1f} miles")

In [ ]:
# Create development likelihood score (0-100)
# Factors:
# - High ridership = more demand (40 points)
# - Close to Center City = higher land values (40 points)
# - (Future: zoning permissiveness, recent permits, etc.)

# Normalize ridership (0-40 scale)
if 'avg_ridership' in stations_ridership.columns:
    max_ridership = stations_ridership['avg_ridership'].max()
    stations_ridership['ridership_score'] = (stations_ridership['avg_ridership'] / max_ridership * 40).fillna(0)
else:
    stations_ridership['ridership_score'] = 20  # Default middle score

# Normalize distance (0-40 scale, inverted - closer is better)
max_dist = stations_ridership['dist_from_cc_miles'].max()
stations_ridership['proximity_score'] = ((max_dist - stations_ridership['dist_from_cc_miles']) / max_dist * 40)

# Total development score
stations_ridership['development_score'] = (
    stations_ridership['ridership_score'] + 
    stations_ridership['proximity_score']
).round(1)

# Categorize
stations_ridership['development_likelihood'] = pd.cut(
    stations_ridership['development_score'],
    bins=[0, 30, 60, 100],
    labels=['Lower', 'Moderate', 'High']
)

print("\n=== DEVELOPMENT LIKELIHOOD DISTRIBUTION ===")
print(stations_ridership['development_likelihood'].value_counts().sort_index())

In [ ]:
# Show stations by development likelihood
display_cols = [station_key, 'DISTRICT', 'avg_ridership', 'dist_from_cc_miles', 
                'development_score', 'development_likelihood']
display_cols = [col for col in display_cols if col in stations_ridership.columns]

print("\n=== HIGH DEVELOPMENT LIKELIHOOD STATIONS ===")
high_dev = stations_ridership[stations_ridership['development_likelihood'] == 'High'].sort_values(
    'development_score', ascending=False
)
print(high_dev[display_cols].to_string(index=False))

# Save comprehensive station analysis
stations_ridership[display_cols].to_csv(OUTPUTS / 'stations_development_analysis.csv', index=False)
print("\n✓ Saved to outputs/stations_development_analysis.csv")

## 6. Interactive Map

In [ ]:
# Create interactive Folium map
# Center on Philadelphia
m = folium.Map(
    location=[39.9526, -75.1652],
    zoom_start=11,
    tiles='CartoDB positron'
)

# Add City Council district boundaries
folium.GeoJson(
    council_districts.to_crs('EPSG:4326'),
    name='City Council Districts',
    style_function=lambda x: {
        'fillColor': 'lightblue',
        'color': 'blue',
        'weight': 2,
        'fillOpacity': 0.1
    },
    tooltip=folium.GeoJsonTooltip(fields=['DISTRICT'], aliases=['District:'])
).add_to(m)

# Add stations with color coding by development likelihood
color_map = {
    'High': 'green',
    'Moderate': 'orange',
    'Lower': 'red'
}

stations_map = stations_ridership.to_crs('EPSG:4326') if stations_ridership.crs != 'EPSG:4326' else stations_ridership

for idx, row in stations_map.iterrows():
    color = color_map.get(row['development_likelihood'], 'gray')
    
    popup_text = f"""
    <b>{row[station_key]}</b><br>
    District: {row['DISTRICT']}<br>
    Avg Daily Ridership: {row['avg_ridership']:,.0f}<br>
    Development Score: {row['development_score']:.1f}/100<br>
    Likelihood: {row['development_likelihood']}
    """ if 'avg_ridership' in row and pd.notna(row['avg_ridership']) else f"""
    <b>{row[station_key]}</b><br>
    District: {row['DISTRICT']}<br>
    Development Score: {row['development_score']:.1f}/100
    """
    
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        popup=folium.Popup(popup_text, max_width=250),
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.7,
        weight=2
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 220px; height: 140px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><b>Development Likelihood</b></p>
<p><i class="fa fa-circle" style="color:green"></i>&nbsp;High (Score 60-100)</p>
<p><i class="fa fa-circle" style="color:orange"></i>&nbsp;Moderate (Score 30-60)</p>
<p><i class="fa fa-circle" style="color:red"></i>&nbsp;Lower (Score 0-30)</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Save map
m.save(OUTPUTS / 'toc_political_map.html')
print("✓ Interactive map saved to outputs/toc_political_map.html")
m

## 7. Political Summary Report

In [ ]:
# Generate summary statistics for political briefing
summary_stats = {
    'Total Regional Rail Stations': len(stations_ridership),
    'City Council Districts with Stations': stations_ridership['DISTRICT'].nunique(),
    'Districts with 0 Stations': 10 - stations_ridership['DISTRICT'].nunique(),
    'High Development Likelihood Stations': (stations_ridership['development_likelihood'] == 'High').sum(),
    'Moderate Development Likelihood': (stations_ridership['development_likelihood'] == 'Moderate').sum(),
    'Lower Development Likelihood': (stations_ridership['development_likelihood'] == 'Lower').sum(),
}

if 'avg_ridership' in stations_ridership.columns:
    summary_stats['Total Daily Ridership (All Stations)'] = f"{stations_ridership['avg_ridership'].sum():,.0f}"
    summary_stats['Average Station Ridership'] = f"{stations_ridership['avg_ridership'].mean():,.0f}"

print("\n" + "="*60)
print("PHILADELPHIA TOC POLITICAL ANALYSIS - SUMMARY")
print("="*60)
for key, value in summary_stats.items():
    print(f"{key:.<50} {value}")
print("="*60)

# Save summary
pd.DataFrame([summary_stats]).T.to_csv(OUTPUTS / 'summary_statistics.csv', header=['Value'])
print("\n✓ Summary saved to outputs/summary_statistics.csv")

In [ ]:
# District-level summary for political targeting
district_political_summary = stations_ridership.groupby('DISTRICT').agg({
    station_key: 'count',
    'avg_ridership': ['sum', 'mean'] if 'avg_ridership' in stations_ridership.columns else station_key,
    'development_likelihood': lambda x: (x == 'High').sum()
}).round(0)

district_political_summary.columns = ['Total_Stations', 'Total_Daily_Ridership', 'Avg_Station_Ridership', 'High_Dev_Stations'] \
    if 'avg_ridership' in stations_ridership.columns else ['Total_Stations', 'High_Dev_Stations']

# Merge with councilmember names
district_political_summary = district_political_summary.reset_index().merge(
    district_info, on='DISTRICT', how='left'
)

print("\n=== DISTRICT POLITICAL TARGETING SUMMARY ===")
print(district_political_summary.to_string(index=False))

# Save
district_political_summary.to_csv(OUTPUTS / 'district_political_summary.csv', index=False)
print("\n✓ Saved to outputs/district_political_summary.csv")

## 8. Key Insights for Political Strategy

Based on this analysis, key points for City Council engagement:

1. **Station Distribution**: Identify which councilmembers represent areas with multiple transit stations
2. **High-Impact Zones**: Focus on high-ridership stations where TOC would affect the most residents
3. **Development Opportunity**: Highlight stations with high development likelihood scores
4. **Equity Considerations**: Note districts with fewer/no stations that might need different policy approaches

### Outputs Generated:
- `stations_per_district.csv` - Station counts by district
- `stations_development_analysis.csv` - Full station-level analysis
- `district_political_summary.csv` - District-level political briefing
- `toc_political_map.html` - Interactive map for presentations
- `stations_per_district_chart.png` - Chart for briefing materials